# Hanoi (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_python.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Hanoi`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Hanoi_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## Torri di Hanoi: Hanoi

**Input:** il numero di dischi $n \ge 1$, il piolo di partenza $from$, quello di arrivo $to$ e quello ausiliario $aux$. **Output:** la sequenza delle mosse che porta i dischi $1, \dots, n$ da $from$ a $to$. $M(n) = 2^n - 1$ mosse (ottime), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ passi, profondità $n$.

Pseudocodice: sezione 14.8 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def move(disk, frm, to):                           # "move disk d from from to to": conta la mossa e, se show_moves, la stampa
    global moves
    moves += 1
    if show_moves:
        print("%d. sposta il disco %d da %s a %s" % (moves, disk, frm, to))


show_moves = False


def hanoi(n, frm, to, aux):                        # in Python "from" e' una parola riservata: frm
    push(n); cnt[2] += 1
    call = "Hanoi(%d,%s,%s,%s)" % (n, frm, to, aux) if trace else ""
    if n == 1:
        cnt[3] += 1
        move(1, frm, to)
        if trace: say(call, "stampa: sposta il disco 1 da %s a %s, ritorna" % (frm, to))
    else:
        if trace: say(call, "chiama Hanoi(%d,%s,%s,%s)" % (n - 1, frm, aux, to))
        cnt[5] += 1
        hanoi(n - 1, frm, aux, to)
        cnt[6] += 1
        move(n, frm, to)
        if trace: say(call, "stampa: sposta il disco %d da %s a %s, chiama Hanoi(%d,%s,%s,%s)" % (n, frm, to, n - 1, aux, to, frm))
        cnt[7] += 1
        hanoi(n - 1, aux, to, frm)
        if trace: say(call, "ritorna")
    pop()

In [3]:
# Esempio delle dispense: Hanoi(3, A, C, B), le mosse
reset()
show_moves = True
hanoi(3, "A", "C", "B")
show_moves = False
print("%d mosse, %d chiamate, profondita' della ricorsione = %d" % (moves, calls, max_depth))

# La stessa esecuzione, chiamata per chiamata (rientro = profondita' nello stack)
print()
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
hanoi(3, "A", "C", "B")
trace = False
print_steps(2, 3, 5, 6, 7)
print("S(n) = 3 * 2^n - 4 = %d" % (3 * 2 ** 3 - 4))

# Mosse e passi: M(n) = 2^n - 1 e S(n) = 3 * 2^n - 4
print()
print("  n |  mosse | 2^n - 1 | chiamate |   S(n) | 3 * 2^n - 4 | profondita'")
for n in range(1, 11):
    reset()
    hanoi(n, "A", "C", "B")
    print("%3d | %6d | %7d | %8d | %6d | %11d | %d" % (n, moves, 2 ** n - 1, calls, sum(cnt), 3 * 2 ** n - 4, max_depth))

# La leggenda: 64 dischi, una mossa al secondo
print()
print("n = 64: 2^64 - 1 secondi sono piu' di %.1e anni" % ((2.0 ** 64 - 1) / (365.25 * 24 * 3600)))

1. sposta il disco 1 da A a C
2. sposta il disco 2 da A a B
3. sposta il disco 1 da C a B
4. sposta il disco 3 da A a C
5. sposta il disco 1 da B a A
6. sposta il disco 2 da B a C
7. sposta il disco 1 da A a C
7 mosse, 7 chiamate, profondita' della ricorsione = 3

chiamata                    | azione
Hanoi(3,A,C,B)              | chiama Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | stampa: sposta il disco 1 da A a C, ritorna
  Hanoi(2,A,B,C)            | stampa: sposta il disco 2 da A a B, chiama Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | stampa: sposta il disco 1 da C a B, ritorna
  Hanoi(2,A,B,C)            | ritorna
Hanoi(3,A,C,B)              | stampa: sposta il disco 3 da A a C, chiama Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | chiama Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | stampa: sposta il disco 1 da B a A, ritorna
  Hanoi(2,B,C,A)            | stampa: sposta il disco 2 da B a C, chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)

**Prova tu.** Cambia il numero di dischi e conta le mosse.

In [4]:
n = 4        # <-- cambia il numero di dischi
reset()
show_moves = True
hanoi(n, "A", "C", "B")
show_moves = False
print("%d mosse = 2^%d - 1 = %d" % (moves, n, 2 ** n - 1))

1. sposta il disco 1 da A a B
2. sposta il disco 2 da A a C
3. sposta il disco 1 da B a C
4. sposta il disco 3 da A a B
5. sposta il disco 1 da C a A
6. sposta il disco 2 da C a B
7. sposta il disco 1 da A a B
8. sposta il disco 4 da A a C
9. sposta il disco 1 da B a C
10. sposta il disco 2 da B a A
11. sposta il disco 1 da C a A
12. sposta il disco 3 da B a C
13. sposta il disco 1 da A a B
14. sposta il disco 2 da A a C
15. sposta il disco 1 da B a C
15 mosse = 2^4 - 1 = 15


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).